In [ ]:
# ============================================
# Startup Cell: Mount Drive + Import Config
# ============================================

from google.colab import drive
drive.mount("/content/drive")

import sys
import os

# -------------------------------------------------
# Project base directory (Drive)
# -------------------------------------------------
BASE_DRIVE_DIR = "/content/drive/MyDrive/DIP_Project"

# -------------------------------------------------
# Add src/ to Python path and import config
# -------------------------------------------------
sys.path.append(f"{BASE_DRIVE_DIR}/src")

from project_config import *

# -------------------------------------------------
# Basic environment confirmation
# -------------------------------------------------
print("Drive mounted successfully.")
print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"METADATA_DIR: {METADATA_DIR}")
print(f"PROCESSED_DATA_DIR: {PROCESSED_DATA_DIR}")
print(f"LOCAL_WORK_DIR: {LOCAL_WORK_DIR}")


In [ ]:
# ============================================
# Cell 0: Notebook Overview
# ============================================
# Purpose:
#   This notebook verifies and applies spatial feature extraction
#   for a selected dataset subset using metadata-driven image selection
#   and preprocessed grayscale images stored on Google Drive.
#
# Inputs:
#   The notebook expects:
#     - a subset-specific metadata CSV file stored in:
#         data/metadata/
#       such as:
#         train_metadata.csv
#         test_metadata.csv
#     - dataset-specific preprocessed image folders stored in:
#         data/preprocessed/<dataset>/images/
#
# Assumptions:
#   - All images have already been preprocessed.
#   - All images are already grayscale.
#   - All images have already been resized to 256 x 256.
#   - This notebook does NOT perform resizing or grayscale conversion.
#   - Input selection is metadata-driven.
#   - Class labels are taken from metadata, not inferred from filenames.
#   - The notebook uses project_config.py as the central source for
#     directory paths, file names, and shared project constants.
#   - This notebook focuses only on spatial feature extraction.
#
# What the notebook does:
#   Startup Cell:
#     Mount Google Drive, import project_config.py, and initialize
#     the notebook environment.
#
#   Cell 1:
#     Import required libraries for image loading, numerical processing,
#     feature extraction, and visualization.
#
#   Cell 2:
#     Define the selected subset, input metadata path, preprocessed image
#     root directory, output CSV path, and dataset-folder mapping.
#
#   Cell 3:
#     Verify required inputs exist before processing:
#       - confirm the selected metadata CSV exists
#       - confirm the preprocessed root directory exists
#       - confirm metadata has expected columns
#       - validate subset values
#       - validate source_dataset values
#       - resolve and verify a sample image path
#
#   Cell 4:
#     Load the selected sample image and verify its properties,
#     including:
#       - shape
#       - datatype
#       - intensity range
#
#   Cell 5:
#     Define helper functions for spatial feature extraction.
#
#   Cell 6:
#     Compute the spatial feature values for the selected sample image.
#
#   Cell 7:
#     Display intermediate visual results for the sample image that
#     support interpretation of the spatial features.
#
#   Cell 8:
#     Display summary outputs or distributions relevant to the selected
#     spatial features.
#
#   Cell 9:
#     Compare a small set of real and AI-generated images from the
#     selected subset and print their spatial feature values.
#
#   Cell 10:
#     Apply spatial feature extraction to all images identified by
#     the selected metadata CSV and save the results to a subset-specific
#     output CSV.
#
#   Cell 11:
#     Reload and verify the saved spatial feature CSV.
#
# Outputs:
#   Primary outputs:
#     - visual validation of spatial feature behavior
#     - printed spatial feature values for selected images
#     - subset-specific spatial feature CSV
#
# Notes:
#   - This notebook supports exploratory validation and full-subset feature
#     extraction for a selected subset (train or test).
#   - Only spatial features are extracted in this notebook.
#   - Later notebooks merge gradient, spatial, and frequency-domain
#     features into complete feature vectors for classifier training
#     and evaluation.
#   - This notebook is intended to follow the same structural pattern
#     as the other feature-extraction notebooks for consistency across
#     the project pipeline.
#   - This notebook must be executed separately for each subset:
#       1. Set SUBSET_NAME = TRAIN_SUBSET and run all cells to generate:
#            train_spatial_features.csv
#       2. Set SUBSET_NAME = TEST_SUBSET and run all cells to generate:
#            test_spatial_features.csv
#
#   - The training and test feature files must remain strictly separate.
#     The training set is used for model development, while the test set
#     is reserved for final evaluation only.
# ============================================

print("Notebook overview loaded.")



In [ ]:
# ============================================
# Cell 1: Import Required Libraries
# ============================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from tqdm.notebook import tqdm

from PIL import Image
from scipy.stats import entropy

# Spatial feature helpers (likely needed)
from skimage.filters.rank import entropy as local_entropy
from skimage.morphology import disk

print("Libraries imported successfully.")



In [ ]:
# ============================================
# Cell 2: Define Subset and Paths
# ============================================

SUBSET_NAME = TRAIN_SUBSET

if SUBSET_NAME == TRAIN_SUBSET:
    INPUT_METADATA_CSV = os.path.join(METADATA_DIR, TRAIN_METADATA_FILENAME)
    OUTPUT_FEATURES_CSV = os.path.join(METADATA_DIR, "train_spatial_features.csv")
elif SUBSET_NAME == TEST_SUBSET:
    INPUT_METADATA_CSV = os.path.join(METADATA_DIR, TEST_METADATA_FILENAME)
    OUTPUT_FEATURES_CSV = os.path.join(METADATA_DIR, "test_spatial_features.csv")
else:
    raise ValueError(f"Unsupported subset: {SUBSET_NAME}")

# Base directory containing dataset-specific preprocessed image folders
PREPROCESSED_ROOT_DIR = PROCESSED_DATA_DIR

# Map metadata source_dataset values to actual folder names in Drive
SOURCE_DATASET_TO_FOLDER = {
    "DiffusionDB": "DiffusionDB",
    "SDXL_Generated_10K": "SDXL_Generated_10K",
    "Midjourney": "Midjourney",
    "ImageNet_1K_256": "ImageNet_1K_256",
    "MS_COCO_2017": "MS_COCO_2017",
    "OpenImages": "OpenImages",
}

SAMPLE_IMAGE_PATH = None
SAMPLE_IMAGE_NAME = None

print(f"Selected subset: {SUBSET_NAME}")
print(f"Input metadata CSV: {INPUT_METADATA_CSV}")
print(f"Preprocessed root directory: {PREPROCESSED_ROOT_DIR}")
print(f"Output feature CSV: {OUTPUT_FEATURES_CSV}")



In [ ]:
# ============================================
# Cell 3: Verify Inputs and Select Sample Image
# ============================================

required_metadata_columns = [
    "filename",
    "class_label",
    "source_dataset",
    "subset",
]

print("Validating inputs...")

# --------------------------------------------
# Check metadata CSV
# --------------------------------------------
if not os.path.exists(INPUT_METADATA_CSV):
    raise FileNotFoundError(f"Missing metadata CSV: {INPUT_METADATA_CSV}")

# --------------------------------------------
# Check preprocessed root directory
# --------------------------------------------
if not os.path.isdir(PREPROCESSED_ROOT_DIR):
    raise FileNotFoundError(
        f"Missing preprocessed root directory: {PREPROCESSED_ROOT_DIR}"
    )

# --------------------------------------------
# Load metadata
# --------------------------------------------
df_meta = pd.read_csv(INPUT_METADATA_CSV)

# --------------------------------------------
# Validate required columns
# --------------------------------------------
missing_columns = [
    col for col in required_metadata_columns
    if col not in df_meta.columns
]

if missing_columns:
    raise ValueError(f"Missing required metadata columns: {missing_columns}")

# --------------------------------------------
# Validate subset
# --------------------------------------------
if not all(df_meta["subset"] == SUBSET_NAME):
    raise ValueError(
        f"Metadata contains rows outside subset '{SUBSET_NAME}'"
    )

# --------------------------------------------
# Validate source_dataset values
# --------------------------------------------
unknown_sources = [
    s for s in df_meta["source_dataset"].unique()
    if s not in SOURCE_DATASET_TO_FOLDER
]

if unknown_sources:
    raise ValueError(f"Unknown source_dataset values: {unknown_sources}")

# --------------------------------------------
# Helper: build full image path
# --------------------------------------------
def get_image_path(row):
    return os.path.join(
        PREPROCESSED_ROOT_DIR,
        SOURCE_DATASET_TO_FOLDER[row["source_dataset"]],
        "images",
        row["filename"],
    )

# --------------------------------------------
# Select sample image
# --------------------------------------------
sample_row = df_meta.iloc[0]
SAMPLE_IMAGE_NAME = sample_row["filename"]
SAMPLE_IMAGE_PATH = get_image_path(sample_row)

if not os.path.exists(SAMPLE_IMAGE_PATH):
    raise FileNotFoundError(f"Sample image not found: {SAMPLE_IMAGE_PATH}")

# --------------------------------------------
# Minimal confirmation output
# --------------------------------------------
print(f"Rows: {len(df_meta)}")
display(df_meta.head())



In [ ]:
# ============================================
# Cell 4: Load Sample Image and Verify Properties
# ============================================

sample_image = Image.open(SAMPLE_IMAGE_PATH)
sample_array = np.array(sample_image)

print("Sample image loaded successfully.")
print(f"Sample image name: {SAMPLE_IMAGE_NAME}")
print(f"Sample image path: {SAMPLE_IMAGE_PATH}")
print(f"Shape: {sample_array.shape}")
print(f"Data type: {sample_array.dtype}")
print(f"Min intensity: {sample_array.min()}")
print(f"Max intensity: {sample_array.max()}")

plt.figure(figsize=(5, 5))
plt.imshow(sample_array, cmap="gray")
plt.title(f"Sample Image: {SAMPLE_IMAGE_NAME}")
plt.axis("off")
plt.show()



In [ ]:
# ============================================
# Cell 5: Spatial Feature Helper Functions
# ============================================

def safe_entropy_from_hist(hist, eps=1e-12):
    hist = hist.astype(np.float64)
    hist = hist / (hist.sum() + eps)
    hist = np.clip(hist, eps, None)
    return float(entropy(hist, base=2))


def image_hist_entropy(img, bins=256):
    hist, _ = np.histogram(img.ravel(), bins=bins, range=(0, 255))
    return safe_entropy_from_hist(hist)


def compute_local_entropy(img, window_size=9):
    """
    Compute local entropy using the same 32-level quantization idea as the
    original loop-based version, but with an optimized scikit-image
    implementation for efficiency.
    """
    img_uint8 = (img * 255).astype(np.uint8)
    img_q = (img_uint8 // 8).astype(np.uint8)  # 32 quantization levels: 0..31

    local_entropy_map = local_entropy(
        img_q,
        disk(window_size // 2)
    ).astype(np.float32)

    return local_entropy_map


def compute_laplacian(img):
    return cv2.Laplacian(img, cv2.CV_32F)


def patch_variance_stats(img, patch_size=16):
    h, w = img.shape
    h_crop = (h // patch_size) * patch_size
    w_crop = (w // patch_size) * patch_size

    img_c = img[:h_crop, :w_crop]

    patches = img_c.reshape(
        h_crop // patch_size, patch_size,
        w_crop // patch_size, patch_size
    ).transpose(0, 2, 1, 3)

    patch_vars = patches.reshape(-1, patch_size, patch_size).var(axis=(1, 2))

    return float(patch_vars.mean()), float(patch_vars.std())


def compute_noise_residual(img):
    blurred = cv2.GaussianBlur(img, (5, 5), 0)
    residual = img - blurred
    return residual


def extract_spatial_features(img):
    global_entropy = image_hist_entropy(img)

    local_entropy_map = compute_local_entropy(img)
    local_entropy_mean = float(np.mean(local_entropy_map))
    local_entropy_std = float(np.std(local_entropy_map))

    intensity_mean = float(np.mean(img))
    intensity_std = float(np.std(img))

    lap = compute_laplacian(img)
    lap_var = float(np.var(lap))

    patch_mean, patch_std = patch_variance_stats(img)

    residual = compute_noise_residual(img)
    residual_energy = float(np.mean(residual ** 2))

    features = {
        "Global Entropy": global_entropy,
        "Local Entropy Mean": local_entropy_mean,
        "Local Entropy Std": local_entropy_std,
        "Intensity Mean": intensity_mean,
        "Intensity Std": intensity_std,
        "Laplacian Variance": lap_var,
        "Patch Variance Mean": patch_mean,
        "Patch Variance Std": patch_std,
        "Noise Residual Energy": residual_energy,
    }

    return features, local_entropy_map, lap, residual



In [ ]:
# ============================================
# Cell 6: Compute Spatial Components
# ============================================

features, local_entropy_map, lap, residual = extract_spatial_features(sample_array)

patch_var_mean, patch_var_std = patch_variance_stats(sample_array)

print(f"Spatial components computed for sample image: {SAMPLE_IMAGE_NAME}")
print("Local entropy map shape =", local_entropy_map.shape)
print("Laplacian shape         =", lap.shape)
print("Residual shape          =", residual.shape)
print("Patch variance mean     =", patch_var_mean)
print("Patch variance std      =", patch_var_std)



In [ ]:
# ============================================
# Cell 7: Display Spatial Visualization Results
# ============================================

fig, axes = plt.subplots(2, 2, figsize=(10, 10))

axes[0, 0].imshow(sample_array, cmap="gray")
axes[0, 0].set_title(f"Input Image: {SAMPLE_IMAGE_NAME}")
axes[0, 0].axis("off")

im1 = axes[0, 1].imshow(local_entropy_map, cmap="viridis")
axes[0, 1].set_title("Local Entropy Map")
axes[0, 1].axis("off")
plt.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04)

im2 = axes[1, 0].imshow(lap, cmap="gray")
axes[1, 0].set_title("Laplacian Response")
axes[1, 0].axis("off")
plt.colorbar(im2, ax=axes[1, 0], fraction=0.046, pad=0.04)

im3 = axes[1, 1].imshow(residual, cmap="gray")
axes[1, 1].set_title("Noise Residual Image")
axes[1, 1].axis("off")
plt.colorbar(im3, ax=axes[1, 1], fraction=0.046, pad=0.04)

plt.suptitle("Spatial Feature Visualization", fontsize=12)
plt.tight_layout()
plt.show()

print(f"Sample: {SAMPLE_IMAGE_NAME}")
print(f"Patch Variance Mean: {patch_var_mean:.6f}")
print(f"Patch Variance Std : {patch_var_std:.6f}")



In [ ]:
# ============================================
# Cell 8: Plot Spatial Histograms
# ============================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(sample_array.ravel(), bins=64, range=(0, 255))
axes[0].set_title("Image Intensity Histogram")
axes[0].set_xlabel("Intensity")
axes[0].set_ylabel("Count")

axes[1].hist(local_entropy_map.ravel(), bins=64)
axes[1].set_title("Local Entropy Histogram")
axes[1].set_xlabel("Local Entropy")
axes[1].set_ylabel("Count")

axes[2].hist(lap.ravel(), bins=64)
axes[2].set_title("Laplacian Histogram")
axes[2].set_xlabel("Laplacian Value")
axes[2].set_ylabel("Count")

plt.suptitle(f"Spatial Histograms: {SAMPLE_IMAGE_NAME}", fontsize=12)
plt.tight_layout()
plt.show()



In [ ]:
# ============================================
# Cell 9: Compare Real vs AI Images
# ============================================

# Select 2 real and 2 AI images (safe sampling)
df_real = df_meta[df_meta["class_label"] == "rl"].sample(
    n=min(2, len(df_meta[df_meta["class_label"] == "rl"])),
    random_state=42
)

df_ai = df_meta[df_meta["class_label"] == "ai"].sample(
    n=min(2, len(df_meta[df_meta["class_label"] == "ai"])),
    random_state=42
)

sample_df = pd.concat([df_real, df_ai], axis=0).reset_index(drop=True)

print(f"Selected images from {SUBSET_NAME} subset:")
print(sample_df[["filename", "class_label", "source_dataset"]].to_string(index=False))
print("\nClass counts:")
print(sample_df["class_label"].value_counts().to_string())
print(f"\nTotal selected: {len(sample_df)}")

for i, row in sample_df.iterrows():
    fname = row["filename"]
    label = row["class_label"]
    source = row["source_dataset"]

    image_path = get_image_path(row)

    img_sample = np.array(Image.open(image_path))
    features_sample, local_entropy_map_s, lap_s, residual_s = extract_spatial_features(img_sample)

    print("\n============================================")
    print(f"{SUBSET_NAME.upper()} | Image {i+1} of {len(sample_df)}")
    print(f"Filename: {fname}")
    print(f"Label: {label} | Source: {source}")

    for k, v in features_sample.items():
        print(f"{k:25s}: {v:.6f}")

    plt.figure(figsize=(8, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(img_sample, cmap="gray")
    plt.title(f"Image ({label})")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(local_entropy_map_s, cmap="viridis")
    plt.title("Local Entropy")
    plt.axis("off")

    plt.suptitle(fname, fontsize=10)
    plt.tight_layout()
    plt.show()



In [ ]:
# ============================================
# Cell 10: Batch Spatial Feature Extraction
# ============================================

rows = []
skipped = 0

for _, row in tqdm(
    df_meta.iterrows(),
    total=len(df_meta),
    desc=f"{SUBSET_NAME} spatial features"
):
    fname = row["filename"]
    image_path = get_image_path(row)

    try:
        img_batch = np.array(Image.open(image_path))

        features_batch, _, _, _ = extract_spatial_features(img_batch)

        out_row = row.to_dict()
        out_row.update(features_batch)
        rows.append(out_row)

    except Exception as e:
        skipped += 1
        print(f"Skipping {fname}: {e}")

# Create DataFrame
features_df = pd.DataFrame(rows)

# Save to CSV
features_df.to_csv(OUTPUT_FEATURES_CSV, index=False)

# Summary
print(f"Saved: {OUTPUT_FEATURES_CSV}")
print(f"Expected rows: {len(df_meta)}")
print(f"Extracted rows: {len(features_df)}")
print(f"Skipped rows: {skipped}")
print(f"Processed subset: {SUBSET_NAME}")

